# K-Nearest Neighbors Training with Polars

This notebook keeps the flow simple:
- load the prepared parquet files
- separate features and target
- validate the numeric training inputs
- train and evaluate the model
- save the model and CSV outputs


## 1. Imports and Config


In [1]:
from pathlib import Path

import polars as pl
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
import joblib

TRAIN_FILE = "../data/train.parquet"
TEST_FILE = "../data/test.parquet"
MODEL_DIR = "../model"
RESULTS_DIR = "../model/results"
TARGET_COLUMN = "label"
RANDOM_SEED = 42
MODEL_NAME = "knn"

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

MODEL_FILE = Path(MODEL_DIR) / "knn_model.joblib"
RESULTS_FILE = Path(RESULTS_DIR) / "knn_results.csv"
METRICS_FILE = Path(RESULTS_DIR) / "knn_metrics.csv"


## 2. Load the Dataset


In [2]:
for required_file in [TRAIN_FILE, TEST_FILE]:
    if not Path(required_file).exists():
        raise FileNotFoundError(f"Expected parquet file at: {required_file}")

train_df = pl.read_parquet(TRAIN_FILE)
test_df = pl.read_parquet(TEST_FILE)

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
train_df.head()


train shape: (81412, 37)
test shape: (20354, 37)


race,gender,age,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,examide,citoglipton,insulin,glyburide-metformin,change,diabetesMed,admission_type,admission_source,discharge_disposition,readmitted
str,str,f64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64
"""Caucasian""","""Female""",55.0,2,"""Private""","""?""",42,1,18,0,0,0,"""injury and poisoning""","""accidental falls such as slipp…","""diseases of the circulatory sy…",9,"""None""","""Norm""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""Emergency""","""Emergency Room""","""Discharged/transferred to home…",0
"""Caucasian""","""Male""",85.0,3,"""Uninsured""","""?""",55,0,11,0,1,0,"""diseases of the circulatory sy…","""diseases of the circulatory sy…","""diseases of the circulatory sy…",6,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""Steady""","""No""","""Ch""","""Yes""","""Elective""","""Physician Referral""","""Discharged/transferred to home…",1
"""Caucasian""","""Female""",65.0,3,"""Public""","""?""",50,6,16,0,0,0,"""diseases of the circulatory sy…","""diseases of the circulatory sy…","""diseases of the circulatory sy…",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Emergency""","""Emergency Room""","""Discharged to home""",0
"""Caucasian""","""Male""",85.0,13,"""Public""","""?""",18,0,15,0,0,1,"""Persons encountering health se…","""Persons with a condition influ…","""diseases of the blood and bloo…",9,"""None""","""None""","""Down""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""Ch""","""Yes""","""Elective""","""Physician Referral""","""Discharged/transferred to home…",0
"""Caucasian""","""Male""",75.0,4,"""Other""","""?""",69,2,21,0,0,0,"""diseases of the circulatory sy…","""diseases of the respiratory sy…","""diseases of the genitourinary …",9,"""None""","""Norm""","""Steady""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""Steady""","""No""","""No""","""Steady""","""No""","""Ch""","""Yes""","""Emergency""","""Emergency Room""","""Discharged to home""",0


## 3. Separate Features and Target

This notebook uses the prepared train and test parquet files exactly as they come from the upstream split notebook.


In [3]:
if TARGET_COLUMN not in train_df.columns or TARGET_COLUMN not in test_df.columns:
    raise ValueError(f"Both parquet files must include the target column: {TARGET_COLUMN}")

feature_columns = [column_name for column_name in train_df.columns if column_name != TARGET_COLUMN]

X_train = train_df.select(feature_columns)
X_test = test_df.select([column_name for column_name in test_df.columns if column_name != TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN].cast(pl.Int32)
y_test = test_df[TARGET_COLUMN].cast(pl.Int32)

print(feature_columns)
print(X_train.shape)
X_train.head()


ValueError: Both parquet files must include the target column: label

## 4. Validate the Prepared Features

These notebooks expect the upstream split and feature-selection notebook to output numeric, model-ready feature columns.


In [ ]:
test_feature_columns = [column_name for column_name in test_df.columns if column_name != TARGET_COLUMN]

if feature_columns != test_feature_columns:
    raise ValueError(
        "Train and test parquet files must have the same feature columns in the same order."
    )

non_numeric_columns = sorted(
    set(
        [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
        + [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
    )
)

if non_numeric_columns:
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric columns found: {non_numeric_columns}"
    )

print(f"validated {len(feature_columns)} numeric feature columns")


## 5. Train the Model


In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train.to_numpy(), y_train.to_numpy())

model


## 6. Evaluate and Save


In [ ]:
probabilities = model.predict_proba(X_test.to_numpy())[:, 1]
predictions = (probabilities >= 0.5).astype(int)
actuals = y_test.to_numpy()

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
}

metrics_frame = pl.DataFrame([metrics])
results_frame = test_df.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
    ]
)

joblib.dump(model, MODEL_FILE)
metrics_frame.write_csv(METRICS_FILE)
results_frame.write_csv(RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics saved to: {METRICS_FILE}")
print(f"results saved to: {RESULTS_FILE}")
metrics_frame


## 7. Preview Results


In [ ]:
results_frame.head(10)
